In [0]:
# Databricks notebook source
# ============================================================
# 🔶 BRONZE INGESTION — SOURCE 1: HRMS (SAP SuccessFactors)
# ============================================================
# Tables: employee_dimension (upsert), work_experience (append)
# ============================================================

Warning you are using the ipython `%run` line magic. To use the databricks `%run` cell magic make sure that the magic is at the very start of the cell.

In [0]:
%run ../05_utils/common_functions

In [0]:
# CELL 3: Define HRMS config
hrms_config = [
    {
        "source_system": "HRMS",
        "source_path": f"/Volumes/{CATALOG}/{SCHEMA}/hrms_files/employee_dimension.csv",
        "target_table": f"{CATALOG}.{SCHEMA}.hrms_employee_dimension",
        "load_type": "upsert",       # grade, location, status change over time
        "primary_key": "employee_id"
    },
    {
        "source_system": "HRMS",
        "source_path": f"/Volumes/{CATALOG}/{SCHEMA}/hrms_files/work_experience.csv",
        "target_table": f"{CATALOG}.{SCHEMA}.hrms_work_experience",
        "load_type": "append",       # historical data — doesn't change
        "primary_key": "experience_id"
    }
]

print(f"📋 HRMS Config: {len(hrms_config)} tables")
for c in hrms_config:
    print(f"   {c['target_table'].split('.')[-1]:<35} → {c['load_type']}")

📋 HRMS Config: 2 tables
   hrms_employee_dimension             → upsert
   hrms_work_experience                → append


In [0]:
# CELL 4: Execute
from datetime import datetime

start = datetime.now()
print(f"🚀 HRMS INGESTION STARTED: {start.strftime('%H:%M:%S')}")

hrms_results = ingest_to_bronze(hrms_config)

end = datetime.now()
print(f"\n{'═'*65}")
print(f"📊 HRMS INGESTION COMPLETE — {(end-start).total_seconds():.1f}s")
print(f"{'═'*65}")
for r in hrms_results:
    print(f"   {r['status']} {r['table']:<35} {r['mode']:<7} → {r['rows']} rows")

🚀 HRMS INGESTION STARTED: 12:36:59

─────────────────────────────────────────────────────────────────
⏳ [HRMS] hrms_employee_dimension
   Source : /Volumes/hackathon_ltm/bronze/hrms_files/employee_dimension.csv
   Target : hackathon_ltm.bronze.hrms_employee_dimension
   Mode   : UPSERT | Key: employee_id
─────────────────────────────────────────────────────────────────
   📄 Records read: 500
   📝 Table doesn't exist → CREATING
   ✅ CREATED: 500 rows

─────────────────────────────────────────────────────────────────
⏳ [HRMS] hrms_work_experience
   Source : /Volumes/hackathon_ltm/bronze/hrms_files/work_experience.csv
   Target : hackathon_ltm.bronze.hrms_work_experience
   Mode   : APPEND | Key: experience_id
─────────────────────────────────────────────────────────────────
   📄 Records read: 500
   📝 Table doesn't exist → CREATING
   ✅ CREATED: 500 rows

═════════════════════════════════════════════════════════════════
📊 HRMS INGESTION COMPLETE — 24.5s
═════════════════════════════════